# cyxla GPU tests

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nardi/cyxla/blob/main/cyxla/notebooks/colab_gpu_tests.ipynb)

This notebook runs `cyxla` on a CUDA GPU. CPU CI cannot test the CUDA plugin, so this
checks that the plugin loads, that programs of every dtype run on it, that the
`jnp.linalg` kernels work, and that lowering and the disk cache behave as they do on the
CPU.

To use it, pick a GPU runtime under *Runtime → Change runtime type → T4 GPU*, set the
fork and branch below, then choose *Runtime → Run all*. Any failing step stops the run
with its error, and the last cell lists the checks that failed.

JAX never runs inside this notebook's own Python process. The project gets its own
virtual environment, built by `uv` from the committed lockfile, and every `cyxla` step
runs as a subprocess in it. That keeps Colab's preinstalled `jax` and CUDA plugin away
from the locked stack, and the runtime needs no restart. The CPU plugin is not
installed, so the CUDA plugin is the only one `cyxla` finds.

In [ ]:
GITHUB_USERNAME = "nardi"  # @param {type:"string"}
BRANCH_NAME = "main"  # @param {type:"string"}

## Preflight check

The `cuda12` and `cuda13` extras need a CUDA 12 or 13 driver. This checks the driver
version, picks the extra that matches it, makes sure `uv` is available, and defines
`run`, which streams a command's output here and raises when the command fails.

In [ ]:
import re
import shutil
import subprocess


def run(command: str, cwd: str | None = None) -> None:
    """Run a shell command, stream its output here, and raise if it fails."""
    print(f"$ {command}", flush=True)
    process = subprocess.Popen(
        command,
        shell=True,
        cwd=cwd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end="", flush=True)
    if process.wait() != 0:
        raise RuntimeError(f"Command failed with exit code {process.returncode}.")


run("nvidia-smi")
smi_output = subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout
match = re.search(r"CUDA Version:\s*(\d+)\.", smi_output)
cuda_major = int(match.group(1)) if match else None
if cuda_major is None or cuda_major < 12:
    raise RuntimeError(
        f"This runtime has CUDA driver version {cuda_major}, and the CUDA plugin needs "
        "12 or newer. Pick a different GPU runtime."
    )
CUDA_EXTRA = "cuda13" if cuda_major >= 13 else "cuda12"
print(f"CUDA driver major version {cuda_major}: OK, so the extra is {CUDA_EXTRA}")

if shutil.which("uv") is None:
    run("pip install uv")
run("uv --version")

## Clone and install

This clones the branch and runs `uv sync` with the CUDA extra and the `build` and `dev`
groups. The `plugin` group is left out, because the CPU plugin has no release to
download yet and the checks only need the CUDA one. `--locked` installs exactly the
versions in `uv.lock`. Setting `UV_PYTHON` makes this `uv sync` and every later `uv run`
use Python 3.12, which `uv` downloads if needed. The build compiles the Cython module
without line tracing, and the CUDA wheels weigh a few GB, so this step takes a few
minutes. The last command lists the plugins that `cyxla` found.

In [ ]:
import os

os.environ["UV_PYTHON"] = "3.12"
run("rm -rf cyxla-repo")
run(
    f"git clone --depth 1 --single-branch -b {BRANCH_NAME} "
    f"https://github.com/{GITHUB_USERNAME}/cyxla.git cyxla-repo"
)
run(
    "uv sync --locked --no-default-groups --group build --group dev "
    f"--extra {CUDA_EXTRA} --config-setting setup-args=-Dcoverage=false",
    cwd="cyxla-repo/cyxla",
)
run("uv run --no-sync python -m cyxla", cwd="cyxla-repo/cyxla")

## Checks

The checks live in one script, so a broken CUDA stack fails with a short message per
check. Each check prints `PASS` or `FAIL`, and the script exits with an error if any
failed. In order, they check that lowering loads no plugin and compiling loads the CUDA
one, that the runtime reports the CUDA platform and copies buffers instead of aliasing
them, that arrays of all 15 dtypes survive a program in both transfer modes, that
elementwise functions and the `jnp.linalg` kernels match NumPy in 32 and 64 bits, that
lowering follows the device of an array and the default device of JAX, that the disk
cache stores and loads an executable, and that 200 repeated runs agree and record no
PJRT error.

In [ ]:
%%writefile gpu_checks.py
"""Checks cyxla on one platform, and exits with an error if any check fails."""

import os
import sys
import tempfile
import time
import traceback

# JAX would otherwise reserve most of the GPU memory for its own client, and cyxla's
# client needs room as well.
os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")
EXPECTED = sys.argv[1] if len(sys.argv) > 1 else "cuda"
os.environ["CYXLA_PLATFORM"] = EXPECTED
os.environ["CYXLA_CACHE_DIR"] = tempfile.mkdtemp(prefix="cyxla-cache-")

import jax
import jax.numpy as jnp
import numpy as np

import cyxla as cx

jax.config.update("jax_enable_x64", True)

JAX_PLATFORM = "gpu" if EXPECTED in ("cuda", "rocm") else EXPECTED
"""The name that JAX gives the platform."""

DTYPES = [
    np.bool_,
    np.int8,
    np.int16,
    np.int32,
    np.int64,
    np.uint8,
    np.uint16,
    np.uint32,
    np.uint64,
    np.float16,
    jnp.bfloat16,
    np.float32,
    np.float64,
    np.complex64,
    np.complex128,
]
"""Every dtype that buffers can hold."""


def compile_function(function, *arguments, disk_cache=False):
    """Lowers and compiles a function, and returns the executable and its statistics."""
    program = cx.lower(function, arguments)
    return cx.compile_program(program, disk_cache=disk_cache)


def check_lazy_initialization():
    """Lowering loads no plugin, and compiling loads the one for the platform."""
    assert not cx.is_initialized(), "The plugin was loaded before any use."
    shape = jax.ShapeDtypeStruct((2,), jnp.float32)
    program = cx.lower(jnp.sin, [shape])
    assert not cx.is_initialized(), "Lowering loaded a plugin."
    assert program.platform == EXPECTED, program.platform
    cx.compile_program(program, disk_cache=False)
    assert cx.is_initialized(), "Compiling did not load a plugin."


def check_runtime():
    """The loaded plugin is the expected one, and aliases memory only on the CPU."""
    platforms = {plugin.platform for plugin in cx.available_plugins()}
    print("Installed plugins:", sorted(platforms))
    assert EXPECTED in platforms, f"No {EXPECTED} plugin is installed."
    info = cx.runtime_info()
    print(info)
    assert info.platform.lower() == EXPECTED, info.platform
    assert info.aliased_inputs == (EXPECTED == "cpu"), info
    assert info.inplace_outputs == (EXPECTED == "cpu"), info


def sample(dtype):
    """Returns a 2 by 3 array of a dtype, with values that are exact in it."""
    dtype = np.dtype(dtype)
    values = np.arange(6).reshape(2, 3)
    if dtype.kind == "b":
        return (values % 2).astype(dtype)
    if dtype.kind == "c":
        return ((values + 1) * (1 + 2j)).astype(dtype)
    return (values + 1).astype(dtype)


def check_dtypes():
    """An array of each dtype goes through a program unchanged, copied or aliased."""
    failures = []
    for dtype in DTYPES:
        values = sample(dtype)
        try:
            shape = jax.ShapeDtypeStruct(values.shape, values.dtype)
            executable, _ = compile_function(lambda x: (x, x), shape)
            for zero_copy in (True, False):
                for result in executable.run([values], zero_copy=zero_copy):
                    assert result.dtype == values.dtype, result.dtype
                    np.testing.assert_array_equal(result, values)
        except Exception as error:
            failures.append(f"{values.dtype}: {type(error).__name__}: {error}")
        else:
            print(f"{values.dtype}: OK")
    assert not failures, "\n".join(failures)


def check_arithmetic():
    """Elementwise functions match NumPy in 32 and 64 bits."""
    values = np.linspace(-2.0, 2.0, 12).reshape(3, 4)
    for dtype, tolerance in ((np.float32, 1e-5), (np.float64, 1e-12)):
        argument = values.astype(dtype)
        shape = jax.ShapeDtypeStruct(argument.shape, argument.dtype)
        executable, _ = compile_function(
            lambda x: jnp.sin(x) * 2 + jnp.exp(-x), shape
        )
        (result,) = executable.run([argument])
        expected = np.sin(argument) * 2 + np.exp(-argument)
        np.testing.assert_allclose(result, expected, rtol=tolerance, atol=tolerance)
        print(f"{np.dtype(dtype)}: OK")


def check_linear_algebra():
    """The kernels of jnp.linalg match NumPy, which needs the custom calls of jaxlib."""
    rng = np.random.default_rng(0)
    base = rng.normal(size=(4, 4))
    matrix = base @ base.T + 4 * np.eye(4)
    functions = {
        "cholesky": (jnp.linalg.cholesky, np.linalg.cholesky),
        "solve": (
            lambda a: jnp.linalg.solve(a, jnp.arange(4.0)),
            lambda a: np.linalg.solve(a, np.arange(4.0)),
        ),
        "inv": (jnp.linalg.inv, np.linalg.inv),
        "eigh": (lambda a: jnp.linalg.eigh(a)[0], lambda a: np.linalg.eigh(a)[0]),
        "svd": (
            lambda a: jnp.linalg.svd(a, compute_uv=False),
            lambda a: np.linalg.svd(a, compute_uv=False),
        ),
        "qr": (
            lambda a: jnp.abs(jnp.linalg.qr(a)[1]),
            lambda a: np.abs(np.linalg.qr(a)[1]),
        ),
    }
    failures = []
    for dtype, tolerance in ((np.float32, 1e-3), (np.float64, 1e-8)):
        argument = matrix.astype(dtype)
        shape = jax.ShapeDtypeStruct(argument.shape, argument.dtype)
        for name, (function, reference) in functions.items():
            try:
                executable, _ = compile_function(function, shape)
                (result,) = executable.run([argument])
                np.testing.assert_allclose(
                    result, reference(matrix), rtol=tolerance, atol=tolerance
                )
            except Exception as error:
                failures.append(
                    f"{name} in {np.dtype(dtype)}: {type(error).__name__}: {error}"
                )
            else:
                print(f"{name} in {np.dtype(dtype)}: OK")
    assert not failures, "\n".join(failures)


def check_platform_selection():
    """Lowering follows the device of an array and the default device of JAX."""
    device = jax.devices(JAX_PLATFORM)[0]
    array = jax.device_put(jnp.arange(3, dtype=jnp.float32), device)
    program = cx.lower(lambda x: x * 2, [array])
    assert program.platform == EXPECTED, program.platform
    with jax.default_device(device):
        assert cx.lower(lambda x: x * 2, [jnp.zeros(3)]).platform == EXPECTED
    executable, _ = cx.compile_program(program, disk_cache=False)
    (result,) = executable.run([array])
    np.testing.assert_array_equal(result, [0.0, 2.0, 4.0])


def check_disk_cache():
    """A compiled executable is stored, and the second compile loads it."""
    shape = jax.ShapeDtypeStruct((5,), jnp.float32)
    program = cx.lower(lambda x: jnp.cos(x) + x, [shape])
    _, first = cx.compile_program(program)
    executable, second = cx.compile_program(program)
    assert (first.compiled, second.loaded) == (1, 1), (first, second)
    values = np.arange(5, dtype=np.float32)
    (result,) = executable.run([values])
    np.testing.assert_allclose(result, np.cos(values) + values, rtol=1e-5, atol=1e-6)


def check_repeated_runs():
    """Many executions give the same answer, and no PJRT error is recorded."""
    errors = cx.error_info().count
    values = np.linspace(0.0, 1.0, 1000, dtype=np.float32)
    shape = jax.ShapeDtypeStruct(values.shape, values.dtype)
    executable, _ = compile_function(lambda x: jnp.sum(jnp.sqrt(x) * x)[None], shape)
    expected = np.sum(np.sqrt(values) * values)
    runs = 200
    start = time.perf_counter()
    for _ in range(runs):
        (result,) = executable.run([values])
        np.testing.assert_allclose(result[0], expected, rtol=1e-4)
    milliseconds = (time.perf_counter() - start) / runs * 1000
    print(f"{runs} runs took {milliseconds:.3f} ms each.")
    assert cx.error_info().count == errors, cx.error_info()


CHECKS = [
    check_lazy_initialization,
    check_runtime,
    check_dtypes,
    check_arithmetic,
    check_linear_algebra,
    check_platform_selection,
    check_disk_cache,
    check_repeated_runs,
]
"""The checks, in the order they run. The first one needs a plugin that is not loaded yet."""

if __name__ == "__main__":
    print("JAX devices:", jax.devices())
    failed = []
    for check in CHECKS:
        print(f"\n== {check.__name__}: {check.__doc__}", flush=True)
        try:
            check()
        except Exception:
            traceback.print_exc()
            failed.append(check.__name__)
            print("FAIL", flush=True)
        else:
            print("PASS", flush=True)
    print(f"\n{len(CHECKS) - len(failed)} of {len(CHECKS)} checks passed on {EXPECTED}.")
    if failed:
        raise SystemExit(f"Failed: {', '.join(failed)}")

In [ ]:
run(f"uv run --no-sync python {os.path.abspath('gpu_checks.py')} cuda", cwd="cyxla-repo/cyxla")